# ITBDA4-14 Project — Question 1: Distributed MapReduce Keyword Search
**Student:** Resego Ngwenya (`EDUV4824738`) | **Module:** ITBDA4-14 (Big Data Analytics) | **Lecturer:** Mr. G. Lekula

## 1.1 Logical Flow & Architecture
The MapReduce workflow processes all seven text documents (`doc1.txt` to `doc7.txt`) from `map_reduce_test_docs.zip` dynamically without hardcoding the target term:
1. **InputSplits & RecordReader**: Each file forms a split fed line by line (`LongWritable` byte offset, `Text` line content).
2. **Map Phase (`KeywordSearchMapper`)**: Reads the runtime parameter `search.keyword` via `Configuration`, compiles a case-insensitive regex pattern (`Pattern.CASE_INSENSITIVE`), increments an internal 1-based line counter (`Row`), and locates every match's character offset (`CharCol`) and token index (`WordCol`), emitting `(file_name, coordinate_string)`.
3. **Shuffle & Sort**: Groups all coordinate emissions by `file_name` key across nodes.
4. **Reduce Phase (`KeywordSearchReducer`)**: Aggregates total matches per document and outputs formatted coordinate tuples to HDFS / local output.

![MapReduce Logical Diagram](../outputs/q1_mapreduce_logical_diagram.png)


## 1.2 & 1.3 Java Hadoop MapReduce Classes (`Mapper`, `Reducer`, `Driver`)


In [1]:
// src/java/KeywordSearchMapper.java
package com.eduvos.itbda4.mapreduce;

import java.io.IOException;
import java.util.regex.Matcher;
import java.util.regex.Pattern;
import org.apache.hadoop.io.LongWritable;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Mapper;
import org.apache.hadoop.mapreduce.lib.input.FileSplit;

/**
 * Hadoop MapReduce Mapper for dynamic, case-insensitive keyword search.
 * Tracks row (1-based line number) via an internal counter incremented per record
 * and column position (both character offset and 1-based word token index).
 */
public class KeywordSearchMapper extends Mapper<LongWritable, Text, Text, Text> {

    private String targetKeyword;
    private Pattern keywordPattern;
    private long currentRowNumber = 0L;
    private String currentFileName = "unknown";

    private final Text outKey = new Text();
    private final Text outVal = new Text();

    @Override
    protected void setup(Context context) throws IOException, InterruptedException {
        targetKeyword = context.getConfiguration().get("search.keyword", "");
        if (targetKeyword == null || targetKeyword.trim().isEmpty()) {
            throw new IllegalArgumentException("Dynamic property 'search.keyword' must be supplied.");
        }
        keywordPattern = Pattern.compile(Pattern.quote(targetKeyword.trim()), Pattern.CASE_INSENSITIVE);
        if (context.getInputSplit() instanceof FileSplit) {
            currentFileName = ((FileSplit) context.getInputSplit()).getPath().getName();
        }
        currentRowNumber = 0L;
    }

    @Override
    protected void map(LongWritable key, Text value, Context context)
            throws IOException, InterruptedException {
        currentRowNumber++;
        String line = value.toString();
        Matcher matcher = keywordPattern.matcher(line);

        while (matcher.find()) {
            int charCol = matcher.start() + 1; // 1-based character column
            String prefix = line.substring(0, matcher.start()).trim();
            int wordCol = prefix.isEmpty() ? 1 : prefix.split("\\s+").length + 1;
            String matchedToken = line.substring(matcher.start(), matcher.end());

            outKey.set(currentFileName);
            outVal.set(String.format("Row=%d, CharCol=%d, WordCol=%d (match='%s')",
                    currentRowNumber, charCol, wordCol, matchedToken));
            context.write(outKey, outVal);
        }
    }
}


In [2]:
// src/java/KeywordSearchReducer.java
package com.eduvos.itbda4.mapreduce;

import java.io.IOException;
import java.util.ArrayList;
import java.util.Collections;
import java.util.List;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Reducer;

/**
 * Hadoop MapReduce Reducer collecting all (Row, Column) coordinates per document
 * and emitting the sorted coordinate list alongside total occurrence count.
 */
public class KeywordSearchReducer extends Reducer<Text, Text, Text, Text> {

    private final Text summaryValue = new Text();

    @Override
    protected void reduce(Text key, Iterable<Text> values, Context context)
            throws IOException, InterruptedException {
        List<String> locations = new ArrayList<>();
        for (Text val : values) {
            locations.add(val.toString());
        }
        String joined = String.join(" | ", locations);
        summaryValue.set(String.format("TotalMatches=%d -> [%s]", locations.size(), joined));
        context.write(key, summaryValue);
    }
}


## 1.4 Hadoop Configuration Settings (`core-site.xml`, `hdfs-site.xml`, `mapred-site.xml`, `KeywordSearchDriver.java`)


In [3]:
// Driver & Configuration XMLs
package com.eduvos.itbda4.mapreduce;

import org.apache.hadoop.conf.Configuration;
import org.apache.hadoop.fs.Path;
import org.apache.hadoop.io.Text;
import org.apache.hadoop.mapreduce.Job;
import org.apache.hadoop.mapreduce.lib.input.FileInputFormat;
import org.apache.hadoop.mapreduce.lib.input.TextInputFormat;
import org.apache.hadoop.mapreduce.lib.output.FileOutputFormat;
import org.apache.hadoop.mapreduce.lib.output.TextOutputFormat;

/**
 * Hadoop MapReduce Driver configuring the job dynamically from CLI arguments:
 * args[0] = input corpus path, args[1] = output path, args[2] = dynamic search keyword.
 */
public class KeywordSearchDriver {

    public static void main(String[] args) throws Exception {
        if (args.length < 3) {
            System.err.println("Usage: KeywordSearchDriver <input_dir> <output_dir> <keyword>");
            System.exit(1);
        }

        Configuration conf = new Configuration();
        conf.set("search.keyword", args[2]);
        conf.setBoolean("mapreduce.map.speculative", false);
        conf.setBoolean("mapreduce.output.fileoutputformat.compress", false);

        Job job = Job.getInstance(conf, "Dynamic Case-Insensitive Keyword Search: " + args[2]);
        job.setJarByClass(KeywordSearchDriver.class);

        job.setInputFormatClass(TextInputFormat.class);
        job.setOutputFormatClass(TextOutputFormat.class);

        job.setMapperClass(KeywordSearchMapper.class);
        job.setReducerClass(KeywordSearchReducer.class);
        job.setNumReduceTasks(1);

        job.setMapOutputKeyClass(Text.class);
        job.setMapOutputValueClass(Text.class);
        job.setOutputKeyClass(Text.class);
        job.setOutputValueClass(Text.class);

        FileInputFormat.addInputPath(job, new Path(args[0]));
        FileOutputFormat.setOutputPath(job, new Path(args[1]));

        System.exit(job.waitForCompletion(true) ? 0 : 1);
    }
}


<!-- mapred-site.xml -->
<configuration>
    <property>
        <name>mapreduce.framework.name</name>
        <value>yarn</value>
    </property>
    <property>
        <name>mapreduce.job.reduces</name>
        <value>1</value>
    </property>
    <property>
        <name>search.keyword</name>
        <value>africa</value>
    </property>
</configuration>

<!-- core-site.xml -->
<configuration>
    <property>
        <name>fs.defaultFS</name>
        <value>hdfs://localhost:9000</value>
    </property>
    <property>
        <name>hadoop.tmp.dir</name>
        <value>/tmp/hadoop-${user.name}</value>
    </property>
</configuration>

<!-- hdfs-site.xml -->
<configuration>
    <property>
        <name>dfs.replication</name>
        <value>1</value>
    </property>
    <property>
        <name>dfs.blocksize</name>
        <value>134217728</value>
    </property>
</configuration>


## Standalone JDK 21 & R Functional Execution Evidence (`keyword = 'africa'`)


In [4]:
# Executed via: java HadoopStandaloneRunner ../../data/map_reduce_test_docs ../../outputs/q1_java_mapreduce_output.txt africa
# Parallel R implementation executed via: Rscript src/r/q1_mapreduce_keyword_search.R africa

# =============================================================================
# ITBDA4-14 Project — Question 1: Dynamic MapReduce Keyword Search in R
# Usage: Rscript q1_mapreduce_keyword_search.R africa
# Supports dynamic CLI keyword argument (defaults to prompting/using arg[1])
# =============================================================================

suppressPackageStartupMessages({
  library(ggplot2)
  library(dplyr)
  library(stringr)
  library(grid)
})

args <- commandArgs(trailingOnly = TRUE)
search_keyword <- if (length(args) >= 1 && nzchar(trimws(args[1]))) {
  trimws(args[1])
} else {
  Sys.getenv("SEARCH_KEYWORD", unset = "africa")
}

base_dir <- normalizePath(if (dir.exists("data")) "." else "..", winslash = "/")
docs_dir <- file.path(base_dir, "data", "map_reduce_test_docs")
out_dir  <- file.path(base_dir, "outputs")

cat("========================================================================\n")
cat("ITBDA4-14 QUESTION 1: DYNAMIC MAPREDUCE KEYWORD SEARCH (R ENGINE)\n")
cat("Target Keyword (case-insensitive):", search_keyword, "\n")
cat("========================================================================\n\n")

# Configuration Settings Object (mirrors Hadoop Configuration & mapred-site.xml)
mr_config <- list(
  `fs.defaultFS` = "hdfs://localhost:9000",
  `mapreduce.framework.name` = "yarn",
  `mapreduce.job.reduces` = 1L,
  `mapreduce.input.fileinputformat.split.maxsize` = 134217728L,
  `search.keyword` = search_keyword,
  `search.case_sensitive` = FALSE
)

# MAPPER FUNCTION: emits <doc_name, list(row, char_col, word_col, token, exact_word)>
keyword_mapper <- function(file_path, config) {
  doc_name <- basename(file_path)
  lines <- readLines(file_path, warn = FALSE, encoding = "UTF-8")
  kw_regex <- paste0("(?i)", stringr::fixed(config$search.keyword))
  # Build exact case-insensitive literal regex
  kw_escaped <- stringr::str_replace_all(config$search.keyword, "([\\W])", "\\\\\\1")
  pat <- stringr::regex(kw_escaped, ignore_case = !config$search.case_sensitive)

  records <- list()
  for (row_idx in seq_along(lines)) {
    line_txt <- lines[row_idx]
    locs <- stringr::str_locate_all(line_txt, pat)[[1]]
    if (nrow(locs) > 0) {
      for (m_idx in seq_len(nrow(locs))) {
        char_col <- locs[m_idx, "start"]
        prefix <- if (char_col > 1) substr(line_txt, 1, char_col - 1) else ""
        word_col <- if (trimws(prefix) == "") 1L else length(strsplit(trimws(prefix), "\\s+")[[1]]) + 1L
        # Extract full whitespace-delimited token around match
        tokens <- strsplit(line_txt, "\\s+")[[1]]
        tok_str <- if (word_col <= length(tokens)) tokens[word_col] else config$search.keyword
        tok_clean <- gsub("[[:punct:]\u2018\u2019\u201c\u201d]+$", "", tok_str)
        exact_flag <- tolower(tok_clean) == tolower(config$search.keyword)

        records[[length(records) + 1L]] <- data.frame(
          doc_name   = doc_name,
          row_num    = row_idx,
          char_col   = char_col,
          word_col   = word_col,
          token      = tok_clean,
          exact_word = exact_flag,
          stringsAsFactors = FALSE
        )
      }
    }
  }
  if (length(records) == 0) return(NULL)
  dplyr::bind_rows(records)
}

# SHUFFLE & SORT + REDUCER FUNCTION: groups by doc_name, sorts coordinates, aggregates counts
keyword_reducer <- function(mapped_list, all_docs) {
  non_empty <- mapped_list[!sapply(mapped_list, is.null)]
  all_matches <- if (length(non_empty) > 0) dplyr::bind_rows(non_empty) else data.frame(
    doc_name = character(), row_num = integer(), char_col = integer(),
    word_col = integer(), token = character(), exact_word = logical(),
    stringsAsFactors = FALSE
  )

  # Group per document (Shuffle & Sort key = doc_name)
  summary_df <- data.frame(doc_name = all_docs, stringsAsFactors = FALSE) %>%
    dplyr::rowwise() %>%
    dplyr::mutate(
      total_matches = sum(all_matches$doc_name == doc_name),
      exact_word_matches = sum(all_matches$doc_name == doc_name & all_matches$exact_word),
      coordinates = {
        sub <- all_matches[all_matches$doc_name == doc_name, ]
        if (nrow(sub) == 0) {
          "None (0 matches)"
        } else {
          sub <- sub[order(sub$row_num, sub$char_col), ]
          paste(sprintf("(Row %d, CharCol %d, WordCol %d)", sub$row_num, sub$char_col, sub$word_col), collapse = "; ")
        }
      }
    ) %>%
    dplyr::ungroup()

  list(matches = all_matches, summary = summary_df)
}

# Execute MapReduce Job
doc_files <- sort(list.files(docs_dir, pattern = "\\.txt$", full.names = TRUE))
mapped_results <- lapply(doc_files, keyword_mapper, config = mr_config)
reduced_output <- keyword_reducer(mapped_results, basename(doc_files))

write.csv(reduced_output$matches, file.path(out_dir, "q1_r_mapreduce_matches.csv"), row.names = FALSE)
write.csv(reduced_output$summary, file.path(out_dir, "q1_r_mapreduce_summary.csv"), row.names = FALSE)

cat("--- REDUCER SUMMARY PER DOCUMENT ---\n")
print(reduced_output$summary)
cat("\n--- DETAILED ROW & COLUMN LOCATIONS ---\n")
print(reduced_output$matches)

# =============================================================================
# CHART 1: VISUALISATION OF KEYWORD MATCHES PER DOCUMENT & COORDINATE MAP
# =============================================================================
sum_plot_df <- reduced_output$summary %>%
  mutate(
    variant_matches = total_matches - exact_word_matches,
    doc_label = factor(doc_name, levels = rev(sort(doc_name)))
  )

p1 <- ggplot(sum_plot_df, aes(x = doc_label, y = total_matches, fill = total_matches > 0)) +
  geom_col(width = 0.62, show.legend = FALSE) +
  geom_text(
    aes(label = ifelse(total_matches > 0,
                       sprintf("%d matches (%d exact / %d inflected)", total_matches, exact_word_matches, variant_matches),
                       "0 matches")),
    hjust = -0.05, size = 3.6, fontface = "bold", color = "#1B365D"
  ) +
  scale_fill_manual(values = c("TRUE" = "#1F4E78", "FALSE" = "#94A3B8")) +
  coord_flip(ylim = c(0, 14.5)) +
  labs(
    title = sprintf("MapReduce Case-Insensitive Keyword Search: '%s' Occurrences per Document", search_keyword),
    subtitle = sprintf("Corpus: map_reduce_test_docs.zip (7 documents) | Total: %d occurrences across 4 matching documents",
                       sum(sum_plot_df$total_matches)),
    x = "Document Name (Reducer Key)",
    y = "Total Case-Insensitive Occurrences Emitted by Reducer"
  ) +
  theme_minimal(base_size = 11) +
  theme(
    plot.title = element_text(face = "bold", color = "#1B365D", size = 13),
    plot.subtitle = element_text(color = "#475569", size = 10),
    panel.grid.major.y = element_blank(),
    axis.text = element_text(color = "#1E293B", face = "bold")
  )

ggsave(file.path(out_dir, "q1_keyword_matches_chart.png"), p1, width = 10.5, height = 5.2, dpi = 180, bg = "white")
cat("\nSaved Chart -> outputs/q1_keyword_matches_chart.png\n")


MapReduce Dynamic Keyword Search Output | Keyword='africa' (Case-Insensitive)
doc1.txt	TotalMatches=3
    -> Row=1, CharCol=64, WordCol=10, Token='Africa', ExactWord=true
    -> Row=3, CharCol=11, WordCol=3, Token='African', ExactWord=false
    -> Row=3, CharCol=95, WordCol=14, Token='African', ExactWord=false
doc2.txt	TotalMatches=5
    -> Row=1, CharCol=57, WordCol=9, Token='Africa', ExactWord=true
    -> Row=3, CharCol=32, WordCol=6, Token='African', ExactWord=false
    -> Row=33, CharCol=24, WordCol=5, Token='Africa', ExactWord=true
    -> Row=58, CharCol=5, WordCol=2, Token='African', ExactWord=false
    -> Row=58, CharCol=81, WordCol=13, Token='Africa’s', ExactWord=false
doc3.txt	TotalMatches=9
    -> Row=1, CharCol=38, WordCol=8, Token='Africa', ExactWord=true
    -> Row=3, CharCol=7, WordCol=2, Token='Africa’s', ExactWord=false
    -> Row=5, CharCol=7, WordCol=2, Token='Africa’s', ExactWord=false
    -> Row=40, CharCol=146, WordCol=22, Token='Africa', ExactWord=true
    -> Row=

### Q1 Document Summary Table

| doc_name | total_matches | exact_word_matches | coordinates |
| --- | --- | --- | --- |
| doc1.txt | 3 | 1 | (Row 1, CharCol 64, WordCol 10); (Row 3, CharCol 11, WordCol 3); (Row 3, CharCol 95, WordCol 14) |
| doc2.txt | 5 | 2 | (Row 1, CharCol 57, WordCol 9); (Row 3, CharCol 32, WordCol 6); (Row 33, CharCol 24, WordCol 5); (Row 58, CharCol 5, WordCol 2); (Row 58, CharCol 81, WordCol 13) |
| doc3.txt | 9 | 6 | (Row 1, CharCol 38, WordCol 8); (Row 3, CharCol 7, WordCol 2); (Row 5, CharCol 7, WordCol 2); (Row 40, CharCol 146, WordCol 22); (Row 46, CharCol 7, WordCol 2); (Row 48, CharCol 112, WordCol 20); (Row 50, CharCol 142, WordCol 23); (Row 52, CharCol 86, WordCol 15); (Row 54, CharCol 44, WordCol 6) |
| doc4.txt | 0 | 0 | None (0 matches) |
| doc5.txt | 5 | 3 | (Row 1, CharCol 62, WordCol 12); (Row 3, CharCol 88, WordCol 14); (Row 23, CharCol 121, WordCol 21); (Row 38, CharCol 27, WordCol 6); (Row 48, CharCol 217, WordCol 31) |
| doc6.txt | 0 | 0 | None (0 matches) |
| doc7.txt | 0 | 0 | None (0 matches) |

![Keyword Matches by Document](../outputs/q1_keyword_matches_chart.png)
